In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()

In [3]:
ruta = rc.validar_ruta()

In [4]:
ruta

WindowsPath('C:/Users/Dataa/Desktop/VENTAS/VENTA MENSUAL')

In [5]:
ruta / 'file' / 'base2025.xlsx'

WindowsPath('C:/Users/Dataa/Desktop/VENTAS/VENTA MENSUAL/file/base2025.xlsx')

In [6]:
ventas = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\file\BASE VENTAS 2025.xlsx")

In [7]:
ventas.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'AÑO', 'MES', 'DIA', 'CLIENTE',
       'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
       'TASA_CAMBIO', 'TRM', 'TOTAL($)', 'TELEFONO', 'EMAIL', 'PAIS', 'CIUDAD',
       'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS', 'REFERENCIA',
       'ZONA', 'ASESOR COMERCIAL'],
      dtype='object')

In [8]:
ventas['CATEGORÍA'].value_counts()

CATEGORÍA
SHOPIFY               110695
MAYORISTA NV           53734
CALL CENTER            30011
DISTRIBUIDOR           12795
FARMACIA                 827
SURTICOSMETICOS          567
EMPLEADO                 523
KRIKA                    314
COOPIDROGAS              287
CATÁLOGO                 265
HOLE COSMETICS SAS       188
EC                       132
Proveedores               93
ESPECIALIZADAS            70
DO                        43
US                        32
PE                         8
CLENTE                     2
Name: count, dtype: int64

In [9]:
cate = ['MAYORISTA NV','DISTRIBUIDOR' ]

In [12]:
ventas_filtradas = ventas[(ventas['CATEGORÍA'].isin(cate))&(ventas['FECHA_FACTURA']>='2025-07-01')]

In [13]:
ventas_filtradas.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'AÑO', 'MES', 'DIA', 'CLIENTE',
       'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
       'TASA_CAMBIO', 'TRM', 'TOTAL($)', 'TELEFONO', 'EMAIL', 'PAIS', 'CIUDAD',
       'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS', 'REFERENCIA',
       'ZONA', 'ASESOR COMERCIAL'],
      dtype='object')

In [14]:
ventas_filtradas['pederid'] = ventas_filtradas['FECHA_FACTURA'].dt.to_period('M')

C:\Users\Dataa\AppData\Local\Temp\ipykernel_21456\3322799253.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ventas_filtradas['pederid'] = ventas_filtradas['FECHA_FACTURA'].dt.to_period('M')


In [18]:
ventas_filtradas = ventas_filtradas.groupby(['CLIENTE', 'pederid']).agg({'TOTAL($)':'sum'}).reset_index()

In [21]:
ventas_filtradas.columns

Index(['CLIENTE', 'pederid', 'TOTAL($)'], dtype='object')

In [32]:
clientes_promo = ventas_filtradas[ventas_filtradas['TOTAL($)']>=10000000]['CLIENTE'].unique().tolist()

In [34]:
ventas_filtradas = ventas_filtradas[ventas_filtradas['CLIENTE'].isin(clientes_promo)]

In [36]:
ventas_filtradas.pivot_table(
    index='CLIENTE',
    columns='pederid',
    values='TOTAL($)'



).reset_index().fillna(0).to_excel(r"C:\Users\Dataa\Desktop\base.xlsx")

In [2]:

# porcesar base de ventas y notas credito
ventas_procesadas = rc.transformar_base(origen=True)
ruta = rc.validar_ruta()

ruta_clean = ruta / 'CLEAN DATA' 

ruta2 = Path(ventas_procesadas['nombre_archivo'])
ruta_carpeta = ruta_clean / f'VENTAS_{ruta2.stem}.xlsx'
ruta_errores = ruta / 'file' / 'ventas_sin_categoria.xlsx'


ruta_bgta= ruta / 'data' / 'Base_bogota.xlsx'
ruta_zonas= ruta / 'data' / 'zonas.xlsx'
df_bogota= pd.read_excel(ruta_bgta)
zonas = pd.read_excel(ruta_zonas)

ventas_procesadas['Base'] = ventas_procesadas['Base'].merge(zonas, on=['DEPARTAMENTO', 'CATEGORÍA'], how='left')
df_bogota['DOCUMENTO'] = df_bogota['DOCUMENTO'].astype(str)
ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'] = ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'].str.strip()




Archivo de ventas 'septiembre.xlsx' cargado correctamente.
Archivo de notas crédito 'notas_septiembre.xlsx' cargado correctamente.
Archivo consolidado guardado como 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\RAW DATA\PROCESADO\septiembre_procesado.xlsx'.
Facturas afectadas por notas crédito:
(184, 4)
Listado de facturas afectadas guardado como 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\RAW DATA\FACTURAS AFECTADAS\septiembre_facturas_afectadas.xlsx'.
Valores nulos en 'Líneas de factura/Total': 0
Valores nulos en 'Líneas de factura/Fecha de factura': 0
Merge completado correctamente.


In [ ]:
ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'] = pd.to_numeric(
    ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'], errors='coerce'
)
df_bogota['DOCUMENTO'] = pd.to_numeric(df_bogota['DOCUMENTO'])

In [4]:

ventas_procesadas['Base'] = ventas_procesadas['Base'].merge(df_bogota[['DOCUMENTO', 'CATEGORÍA', 'ZONA']], 
                                left_on=['IDENTIFICACION_CLIENTE','CATEGORÍA'], right_on=['DOCUMENTO', 'CATEGORÍA'], how='left')


In [5]:

ventas_procesadas['Base']['ZONA'] = ventas_procesadas['Base']['ZONA'].fillna(ventas_procesadas['Base']['zona'])

ventas_procesadas['Base'] = ventas_procesadas['Base'].drop(columns=['CLIENTES NUEVOS',	'zona',	'DOCUMENTO'])

try:
    ruta_clean.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_clean}' creada o ya existe.")
    ventas_procesadas['Base'].to_excel(ruta_carpeta, index=False)
    with pd.ExcelWriter(ruta_errores, engine='openpyxl') as writer:
        ventas_procesadas['errores'].to_excel(writer, sheet_name='etiqueta a tipo', index=False)
        ventas_procesadas['cliente_call_center'].to_excel(writer, sheet_name='CLIENTE a CALL', index=False)
        ventas_procesadas['asesores_sin_categoria'].to_excel(writer, sheet_name='Mayoristas sin categoria', index=False)
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")
# Consolidar ventas
base_clean = rc.consolidar_carpeta(ruta_carpeta=ruta_clean)
ruta_base = ruta / 'file' / 'BASE VENTAS 2025.xlsx'
import locale
try:         
    # Intentamos usar el locale en español para obtener "ENERO", "FEBRERO", etc.
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except locale.Error:
    print("   - Advertencia: Locale 'es_ES.UTF-8' no disponible. Se usarán nombres de mes en inglés.")
    
base_clean['MES'] = base_clean['FECHA_FACTURA'].dt.strftime('%B').str.upper()
columnas_finales = [
        "Source.Name", "NUMERO_FACTURA", "FECHA_FACTURA", "AÑO", "MES", "DIA",
        "CLIENTE", "IDENTIFICACION_CLIENTE", "CATEGORÍA", "PRODUCTO", "CANTIDAD",
        "TOTAL", "TASA_CAMBIO", "TRM", "TOTAL($)", "TELEFONO", "EMAIL", "PAIS",
        "CIUDAD", "CIUDAD_CORREGIDA", "DEPARTAMENTO", "EQUIPO_VENTAS", "REFERENCIA", "ZONA"
    ]
    
# Manejo defensivo por si la columna 'ASESOR COMERCIAL' no siempre existe
if 'ASESOR COMERCIAL' in base_clean.columns:
    base_clean['ASESOR COMERCIAL'] = base_clean['ASESOR COMERCIAL'].astype(str)
    columnas_finales.append('ASESOR COMERCIAL')

# Aseguramos que solo reordenamos las columnas que realmente existen en el DataFrame
columnas_existentes = [col for col in columnas_finales if col in base_clean.columns]
base_clean = base_clean[columnas_existentes]

# Esta linea mantiene solo los pruductos comerciales
base_clean = base_clean[base_clean['PRODUCTO'].str.startswith(('[PCN','[KD','[TNG','[B8'))]   ###### linea modificada

# ruta_bgta= ruta / 'data' / 'Base_bogota.xlsx'
# ruta_zonas= ruta / 'data' / 'zonas.xlsx'
# df_bogota= pd.read_excel(ruta_bgta)
# zonas = pd.read_excel(ruta_zonas)

# base_clean = base_clean.merge(zonas, on=['DEPARTAMENTO', 'CATEGORÍA'], how='left')
# df_bogota['DOCUMENTO'] = df_bogota['DOCUMENTO'].astype(str)
# base_clean['IDENTIFICACION_CLIENTE'] = base_clean['IDENTIFICACION_CLIENTE'].str.strip()
# base_clean = base_clean.merge(df_bogota[['DOCUMENTO', 'CATEGORÍA', 'ZONA']], 
#                               left_on=['IDENTIFICACION_CLIENTE','CATEGORÍA'], right_on=['DOCUMENTO', 'CATEGORÍA'], how='left')

# base_clean['ZONA'] = base_clean['ZONA'].fillna(base_clean['zona'])


# base_clean = base_clean[['NUMERO_FACTURA', 'FECHA_FACTURA', 'AÑO', 'MES', 'DIA', 'CLIENTE',
#     'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
#     'TASA_CAMBIO', 'TRM', 'TOTAL($)', 'TELEFONO', 'EMAIL', 'PAIS', 'CIUDAD',
#     'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS', 'REFERENCIA',
#     'ASESOR COMERCIAL', 'ZONA']]



try:
    ruta_file = ruta / 'file' 
    ruta_file.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_file}' creada o ya existe.")
    base_clean.to_excel(ruta_base, index=False)
    
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")


explosion = rc.explosion_ventas()



Carpeta 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\CLEAN DATA' creada o ya existe.
Buscando archivos con extensión '.xlsx' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\CLEAN DATA
  - Archivo 'VENTAS_septiembre_procesado.xlsx' leído correctamente.
  - Archivo 'VENTAS_ventas_2024_procesado.xlsx' leído correctamente.
  - Archivo 'VENTAS_ventas_procesado.xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
Carpeta 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\file' creada o ya existe.
Error al crear la carpeta o guardar los archivos: [Errno 13] Permission denied: 'C:\\Users\\Dataa\\Desktop\\VENTAS\\VENTA MENSUAL\\file\\BASE VENTAS 2025.xlsx'
Productos con 'KIT' sin correspondencia en df_kits: ['[PCN15] KIT HYDRO CLEAN HC']
Carpeta 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\file' creada o ya existe.
Archivos guardados correctamente.
